In [6]:
import os
import cv2
import numpy as np
import pandas as pd
from feat.detector import Detector
from moviepy.editor import VideoFileClip
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Masking
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [14]:
# Initialize face detector
detector = Detector(verbose=False)


INFO:root:Loading Face model: retinaface
INFO:root:Loading Facial Landmark model: mobilefacenet
INFO:root:Loading facepose model: img2pose
INFO:root:Loading AU model: xgb
INFO:root:Loading emotion model: resmasknet
INFO:root:Loading Identity model: facenet


In [12]:
# Function to extract frames from video
def extract_frames(video_path, fps=1):
    """
    Extracts frames from the video at a specified FPS (frames per second).
    Args:
        video_path (str): Path to the video file.
        fps (int): Frames per second to sample from the video.

    Returns:
        frames (list): List of frames (as numpy arrays) from the video.
    """
    try:
        video = VideoFileClip(video_path)
        frames = []
        for i, frame in enumerate(video.iter_frames(fps=fps)):
            frames.append(frame)
        return frames
    except Exception as e:
        print(f"Error extracting frames from {video_path}: {e}")
        return []  # Return an empty list if the video cannot be processed

# Function to randomly sample a fixed number of frames from a video
def random_sample_frames(frames, num_samples):
    """
    Randomly sample a fixed number of frames from the video.
    Args:
        frames (list): List of frames from the video.
        num_samples (int): Number of frames to sample.

    Returns:
        sampled_frames (list): Randomly sampled frames.
    """
    if len(frames) == 0:
        return []  # Handle case where no frames are available
    if len(frames) < num_samples:
        # If the video has fewer frames than num_samples, pad by repeating frames
        return np.tile(frames, (num_samples // len(frames) + 1, 1, 1, 1))[:num_samples]
    indices = np.random.choice(len(frames), num_samples, replace=False)
    return [frames[i] for i in indices]

# Function to extract features for both faces in the frame
def extract_features_from_frames(frames):
    """
    Extracts facial features from both the caregiver and the child for each frame.
    Args:
        frames (list): List of frames.

    Returns:
        features_list (np.array): Combined feature arrays for both faces in each frame.
    """
    features_list = []
    for frame in frames:
        # Detect faces and extract features
        detections = detector.detect_frame(frame)
        if detections is not None and len(detections) >= 2:
            # Assume two main faces, caregiver and child
            # Combine their features (for simplicity, take first two faces detected)
            face_1_features = detections.iloc[0].to_numpy()  # Caregiver
            face_2_features = detections.iloc[1].to_numpy()  # Child
            combined_features = np.hstack([face_1_features, face_2_features])
        else:
            # If not enough faces detected, pad with zeros
            combined_features = np.zeros((2 * 229,))  # 229 features per face
        features_list.append(combined_features)

    return np.array(features_list) if features_list else np.zeros((1, 2 * 229))  # Return zero if empty

# Function to prepare the dataset from videos and labels
def prepare_data(video_folder, label_file, num_samples=30, fps=1):
    """
    Prepare the dataset by extracting features from videos and aligning with labels.
    Args:
        video_folder (str): Path to the folder containing videos.
        label_file (str): Path to the Excel file with video IDs and humor labels.
        num_samples (int): Number of frames to sample from each video.
        fps (int): Frames per second for frame extraction.

    Returns:
        X (np.array): Feature array for the videos.
        y (np.array): Label array.
    """
    labels_df = pd.read_excel(label_file)
    X = []
    y = []

    for index, row in labels_df.iterrows():
        video_id = row['VideoID']
        humor_label = row['HowFunny']

        video_path = os.path.join(video_folder, f"{video_id}.mp4")

        if os.path.exists(video_path):
            # Extract frames from the video
            frames = extract_frames(video_path, fps=fps)

            # Randomly sample frames
            sampled_frames = random_sample_frames(frames, num_samples)

            if len(sampled_frames) == 0:
                print(f"No valid frames for video {video_id}, skipping.")
                continue  # Skip videos with no valid frames

            # Extract features from both faces
            features = extract_features_from_frames(sampled_frames)

            # Append to dataset
            X.append(features)
            y.append(humor_label)
        else:
            print(f"Video {video_id} not found.")
    
    return np.array(X), np.array(y)

# Encode labels using one-hot encoding for multiclass classification
def encode_labels(y):
    """
    Encodes categorical labels into one-hot numeric format for multiclass classification.
    Args:
        y (list or np.array): Label array.

    Returns:
        encoded_labels (np.array): One-hot encoded labels.
    """
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    return to_categorical(y_encoded)

# Build a sequential LSTM model for multiclass classification
def build_lstm_model(input_shape, num_classes):
    """
    Build a sequential LSTM model to classify humor in a multiclass setting.
    Args:
        input_shape (tuple): Shape of the input (num_samples, num_features).
        num_classes (int): Number of humor classes.

    Returns:
        model (tf.keras.Model): Compiled LSTM model for multiclass classification.
    """
    model = Sequential()
    model.add(Masking(mask_value=0.0, input_shape=input_shape))  # Mask padded values
    model.add(LSTM(64))
    model.add(Dense(num_classes, activation='softmax'))  # Softmax for multiclass classification
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Main script to run the pipeline
if __name__ == '__main__':
    # Define paths and parameters
    video_folder = 'data/videos'  # Folder containing videos
    label_file = 'data/excel.xlsx'  # Excel file with labels
    num_samples = 30  # Number of frames to sample from each video
    fps = 1  # Frames per second to extract from video

    # Prepare data
    X, y = prepare_data(video_folder, label_file, num_samples, fps)

    # Encode labels using one-hot encoding
    y = encode_labels(y)

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Build the LSTM model
    input_shape = (num_samples, X_train.shape[2])  # Input shape for LSTM (num_samples, num_features)
    num_classes = y_train.shape[1]  # Number of humor classes (one-hot encoded dimension)
    model = build_lstm_model(input_shape, num_classes)

    # Train the model
    model.fit(X_train, y_train, epochs=10, batch_size=16, validation_data=(X_test, y_test))

    # Evaluate the model
    loss, accuracy = model.evaluate(X_test, y_test)
    print(f"Test Loss: {loss}, Test Accuracy: {accuracy}")

Video NFCK3A.joke1.rep1.take1.ThatsNotAHat.mp4 not found.
Video NFCK3A.joke1.rep1.take2.ThatsNotAHat.mp4 not found.
Video NFCK3A.joke1.rep2.take1.ThatsNotAHat.mp4 not found.
Video NFCK3A.joke1.rep3.take1.ThatsNotAHat.mp4 not found.
Video NFCK3A.joke2.rep1.take1.TearingPaper.mp4 not found.
Video NFCK3A.joke2.rep2.take1.TearingPaper.mp4 not found.
Video NFCK3A.joke2.rep3.take1.TearingPaper.mp4 not found.
Video NFCK3A.joke3.rep1.take1.NomNomNom.mp4 not found.
Video NFCK3A.joke3.rep2.take1.NomNomNom.mp4 not found.
Video NFCK3A.joke3.rep3.take1.NomNomNom.mp4 not found.
Video NFCK3A.joke4.rep1.take1.ThatsNotACat.mp4 not found.
Video NFCK3A.joke4.rep2.take1.ThatsNotACat.mp4 not found.
Video NFCK3A.joke4.rep3.take1.ThatsNotACat.mp4 not found.
Video NFCK3A.joke5.rep1.take1.Peekaboo.mp4 not found.
Video NFCK3A.joke5.rep2.take1.Peekaboo.mp4 not found.
Video NFCK3A.joke5.rep3.take1.Peekaboo.mp4 not found.
Video GKW7LD.joke1.rep1.take1.ThatsNotAHat.mp4 not found.
Video GKW7LD.joke1.rep2.take1.Thats

ValueError: zero-size array to reduction operation maximum which has no identity